In [ ]:
import pandas as pd
from scipy.stats import pearsonr
import os
from scipy.signal import correlate

#pearson n deu um bom resultado :/ 

def get_dataset_name(file_path):
    return os.path.splitext(os.path.basename(file_path))[0]

def calculate_correlations_for_datasets(origin_substring, folder_path):
    # Filtra arquivos na pasta que correspondem à substring e possuem '-' no nome
    files = [f for f in os.listdir(folder_path) if f.endswith('.csv') and origin_substring in f and '-' in f.lower()]
    
    # Separar os datasets por origem
    datasets = {}
    
    # Carregar e armazenar os datasets
    for file in files:
        dataset_path = os.path.join(folder_path, file)
        dataset_name = get_dataset_name(dataset_path)
        
        # Carregar o dataset
        dataset = pd.read_csv(dataset_path)
        
        # Limpar e preparar o dataset
        dataset['Timestamp_cubic'] = pd.to_datetime(dataset['Timestamp_cubic'], unit='s')
        
        # Armazenar o dataset
        datasets[dataset_name] = dataset
    
    # Gerar o CSV de correlações para cada origem
    correlation_results = []

    # Obter todas as colunas de interesse para correlação
    all_columns = set()
    for dataset in datasets.values():
        all_columns.update(dataset.columns)
    
    # Remover Timestamp_cubic da lista de colunas de interesse
    all_columns.discard('Timestamp_cubic')
    all_columns.discard('Link_bottleneck')

    # Iterar pelos pares de datasets e calcular correlação para as colunas de interesse
    for name1, dataset1 in datasets.items():
        for name2, dataset2 in datasets.items():
            if name1 != name2:
                # Merge dos datasets com base no Timestamp
                merged_df = pd.merge_asof(dataset1.sort_values('Timestamp_cubic'),
                                          dataset2.sort_values('Timestamp_cubic'),
                                          on='Timestamp_cubic',
                                          direction='nearest',
                                          tolerance=pd.Timedelta('1 hours'))

                # Inicializa o dicionário para armazenar as correlações
                link1 = name1.split('-')[2]+"-"+name1.split('-')[3]
                link2 = name2.split('-')[2]+"-"+name2.split('-')[3]
                correlation_data = {
                    'Dataset1': link1,
                    'Dataset2': link2
                }

                # Calcular correlação para as colunas de interesse
                for col in all_columns:
                    if col in dataset1.columns and col in dataset2.columns:
                        merged_df[col + '_x'] = pd.to_numeric(merged_df[col + '_x'], errors='coerce')
                        merged_df[col + '_y'] = pd.to_numeric(merged_df[col + '_y'], errors='coerce')

                        # Verificar se há dados suficientes para calcular a correlação
                        valid_data = merged_df.dropna(subset=[col + '_x', col + '_y'])
                        
                        if len(valid_data) > 1:
                            # Verificar se há variação nos dados antes de calcular a correlação
                            if valid_data[col + '_x'].var() != 0 and valid_data[col + '_y'].var() != 0:
                                try:
                                    corr, _ = pearsonr(valid_data[col + '_x'], valid_data[col + '_y'])
                                except ValueError:
                                    corr = None
                            else:
                                corr = None
                        else:
                            corr = None
                        
                        # Adicionar a correlação ao dicionário
                        correlation_data[col] = corr
                    else:
                        # Se a coluna não estiver presente em um dos datasets
                        correlation_data[col] = None
                
                # Adicionar os dados de correlação para esse par de datasets
                correlation_results.append(correlation_data)
    
    # Criar o DataFrame com os resultados
    correlation_df = pd.DataFrame(correlation_results)
    
    # Verificar se há correlação nos resultados
    #print(correlation_df.head())  # Para depuração

    # Salvar arquivo com o nome baseado na origem (substring)
    #correlation_df.to_csv(f'correlations_{origin_substring}.csv', index=False)
    interessa = correlation_df.drop(columns = ['Dataset1', 'Dataset2'])
    media = interessa.mean()
    print(f'media dos datsets com mesma origem {origin_substring}: {media}')
    #print(f"Correlação calculada e salva no arquivo correlations_{origin_substring}.csv com sucesso.")

# Defina o caminho da pasta
pasta = '../../datasets/serie-multivariada'
lista=[]
# Percorra todos os arquivos na pasta
for nomearquivo in os.listdir(pasta):
    # Verifique se o arquivo é um CSV
    if nomearquivo.endswith('.csv'):
        # Extraia o local (parte do nome do arquivo)
        local = nomearquivo.split('-')[2]
        if local in lista:
            pass
        else:
            lista.append(local)

lista2 = ['ms', 'rs', 'rn', 'mt', 'pi', 'es', 'rr', 'am', 'to', 'ro', 'pb', 'ac', 'ce', 'se', 'ba', 'go', 'ap', 'ma']
for source in lista2:
    s = "-" + source + "-"
    calculate_correlations_for_datasets(s, '../../datasets/serie-multivariada')
